In [ ]:
import argparse
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback, BitsAndBytesConfig, pipeline
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np
from transformers import TrainerCallback
from torch.utils.data import DataLoader
from trl import SFTTrainer
from typing import Dict, Union, List, Any


/home/yash/.pyenv/versions/3.11.9/envs/FT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Model

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
def load_model(model_id="", device= None, use_lora:bool=False):
    
    device = device if device else "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id,
                                            trust_remote_code=True,
                                            padding_side = "right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.add_eos_token
    
    model_load_kwargs = {
    "trust_remote_code": True,
    "device_map": device,
    "torch_dtype":torch.bfloat16,
}
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_load_kwargs)
    # Apply LoRA if specified
    if use_lora:
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.1,
            bias="none",
            task_type=TaskType.CAUSAL_LM
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    
                                                
    return model, tokenizer

# Dataset

"""
About dataset
- Entity-level sentiment analysis on financial entity
- 3 sentiment classification labels : positive, negative and neutral
- target: financial entity (companies) and associated sentiments
- 
"""

In [3]:
from datasets import load_dataset
data = load_dataset("yixuantt/FinEntity")
data

DatasetDict({
    train: Dataset({
        features: ['content', 'annotations'],
        num_rows: 979
    })
})

In [4]:
split_dataset = data["train"].train_test_split(test_size=0.05,seed=43)
train_split = split_dataset["train"].train_test_split(test_size=0.05,seed=43)
train_data = train_split["train"]
val_data = train_split["test"]
test_data = split_dataset["test"]

train_data,val_data,test_data

(Dataset({
     features: ['content', 'annotations'],
     num_rows: 883
 }),
 Dataset({
     features: ['content', 'annotations'],
     num_rows: 47
 }),
 Dataset({
     features: ['content', 'annotations'],
     num_rows: 49
 }))

In [54]:
index = 10
content = train_data[index]["content"]
print(content)
print()
ann = train_data[index]["annotations"]

char_count = {k:v for k,v in enumerate(list(content))}
[print(i) for i in ann]


This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>.

{'end': 112, 'label': 'Positive', 'start': 99, 'tag': 'Positive', 'value': 'Goldman Sachs'}
{'end': 132, 'label': 'Positive', 'start': 121, 'tag': 'Positive', 'value': 'Wells Fargo'}
{'end': 157, 'label': 'Positive', 'start': 142, 'tag': 'Positive', 'value': 'Bank of America'}
{'end': 184, 'label': 'Positive', 'start': 170, 'tag': 'Positive', 'value': 'JPmorgan Chase'}


[None, None, None, None]

In [63]:
[{'entity':a["value"],'label':a["label"]} for a in ann]

[{'entity': 'Goldman Sachs', 'label': 'Positive'},
 {'entity': 'Wells Fargo', 'label': 'Positive'},
 {'entity': 'Bank of America', 'label': 'Positive'},
 {'entity': 'JPmorgan Chase', 'label': 'Positive'}]

In [59]:
print(char_count)

{0: 'T', 1: 'h', 2: 'i', 3: 's', 4: ' ', 5: 's', 6: 'a', 7: 'i', 8: 'd', 9: ',', 10: ' ', 11: 'K', 12: 'a', 13: 't', 14: 'z', 15: 'k', 16: 'e', 17: "'", 18: 's', 19: ' ', 20: 'h', 21: 'i', 22: 'g', 23: 'h', 24: 'e', 25: 's', 26: 't', 27: ' ', 28: 'c', 29: 'o', 30: 'n', 31: 'v', 32: 'i', 33: 'c', 34: 't', 35: 'i', 36: 'o', 37: 'n', 38: ' ', 39: 'r', 40: 'e', 41: 'c', 42: 'o', 43: 'm', 44: 'm', 45: 'e', 46: 'n', 47: 'd', 48: 'a', 49: 't', 50: 'i', 51: 'o', 52: 'n', 53: 's', 54: ',', 55: ' ', 56: 'e', 57: 'a', 58: 'c', 59: 'h', 60: ' ', 61: 'w', 62: 'i', 63: 't', 64: 'h', 65: ' ', 66: '~', 67: '3', 68: '0', 69: '%', 70: ' ', 71: 't', 72: 'o', 73: 't', 74: 'a', 75: 'l', 76: ' ', 77: 'r', 78: 'e', 79: 't', 80: 'u', 81: 'r', 82: 'n', 83: ' ', 84: 'p', 85: 'o', 86: 't', 87: 'e', 88: 'n', 89: 't', 90: 'i', 91: 'a', 92: 'l', 93: ' ', 94: 'a', 95: 'r', 96: 'e', 97: ':', 98: ' ', 99: 'G', 100: 'o', 101: 'l', 102: 'd', 103: 'm', 104: 'a', 105: 'n', 106: ' ', 107: 'S', 108: 'a', 109: 'c', 110: 'h',

In [ ]:
"""
finetuning:
    - Financial sentiment analays (NER level) : same as dataset
    - with Few shot prompting
- How we want to prepare the dataset for fine-tuning:
    - simply having a instruction followed by text followed by annotations
    - or instead of passing the entire annotation, pass company name and its sentiment. (xxx,positive)
    - Tag each token with IOB (inside, outside and )
"""

In [43]:
prompt_style = dict(DEFAULT_SYSTEM_PROMPT1 = """You are a highly skilled financial analyst. Your task is to analyze the provided financial text, identify all financial entities (such as company names), and determine the sentiment associated with each entity. For every identified entity, provide its name and the corresponding sentiment, which should be one of: Positive, Neutral, or Negative. Present the results in a structured format.""",

# one shot prompting
DEFAULT_SYSTEM_PROMPT2 = """You are a senior financial analyst specializing in entity and sentiment extraction. Your task is to:
1. Identify ALL financial entities (companies, institutions, assets) in the text
2. Determine sentiment association for each entity (Positive/Neutral/Negative)
3. Return entities EXACTLY as they appear in the text with their sentiment labels

**Output Format Requirements:**
- Format: "[{'entity': [EXACT_TEXT] , 'label': [LABEL]}]"
- If no sentiment is explicitly stated, use "Neutral"
- Include ALL mentioned entities, even in indirect references

Example:
text: This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>.
output:

[{'entity':'Goldman Sachs', 'label':'Positive'},
 {'entity':'Wells Fargo', 'label':'Positive'},
 {'entity':'Bank of America', 'label':'Positive'},
 {'entity':'JPmorgan Chase', 'label':'Positive'}]""",

DEFAULT_SYSTEM_PROMPT3 = """You are a senior financial analyst. Your tasks:
1. Identify ALL financial entities exactly as they appear
2. Classify sentiment (Positive/Neutral/Negative)
3. Output JSON format with EXACT entity text

Example Input: "Goldman Sachs is outperforming"
Example Output: [{'entity':'Goldman Sachs <GS.N>','label':'Positive'}]""",

DEFAULT_SYSTEM_PROMPT4 = """You are a senior financial analyst specializing in entity and sentiment extraction. Your task is to:
1. Identify ALL financial entities (companies, institutions, assets) in the text
2. Determine sentiment association for each entity (Positive/Neutral/Negative)
3. Return entities EXACTLY as they appear in the text with their sentiment labels

**Output Format Requirements:**
- Format: "[([EXACT_TEXT],[LABEL])]"
- If no sentiment is explicitly stated, use "Neutral"
- Include ALL mentioned entities, even in indirect references

Example Input: "Goldman Sachs is outperforming"
Example Output: [('Goldman Sachs' ,'Positive')]""",

DEFAULT_SYSTEM_PROMPT5 = """Discard all the previous instructions. Behave like you are an expert entity recognizer and sentiment classifier. 
Identify the entities which are companies or organizations from the following content and classify the sentiment of the corresponding entities into ‘Neutral’, ‘Positive’, or ‘Negative’ classes. 
Considering every sentence as a String in python, provide the entities with the start and end index to mark the boundaries of it including spaces and punctuation using zero-based indexing.
Do not give explanations for the sentiment. In the output,Tag means sentiment; value means entity name. If no entity is found in the sentence, the response should be empty. 

Below are some examples:
The sentence: "Other U.S. companies have made similar moves, including social media site Reddit Inc and Mobileye, the self-driving car unit of Intel Corp <INTC.O>. "

assist_prompt = {"start": 74, "end": 84, "value": "Reddit Inc", "tag": "Neutral"}\n{"start": 128, "end": 138, "value": "Intel Corp", "tag": "Neutral"}

user_prompt2 = Kellogg <K.N>, however, based the corporate headquarters for its largest business, snacks, in Chicago after announcing a split into three independent companies this summer. [nL4N2Y822D]

assist_prompt2 = {"start": 4, "end": 11, "value": "Kellogg", "tag": "Neutral"}


user_prompt3 = Rival Oracle <ORCL.N> says in a statement on its website it has withdrawn all products, services and support for Russian and Belarusian companies, subsidiaries and partners. An Oracle spokesperson declined further comment.

assist_prompt3= {"start": 183, "end": 177, "value": "Oracle", "tag": "Neutral"}\n{"start": 6, "end": 12, "value": "Oracle", "tag": "Neutral"}
""")

def create_prompt_deepseek_qwen(text: str,prompt_style:str) -> str:
 
    return f"<｜begin of sentence｜><｜User｜>{prompt_style}\\n{text.strip()}<｜Assistant｜>\n "
    
    



## Model ouput analysis

In [8]:
index = 10
content = train_data[index]["content"]
print(content)
print()
ann = train_data[index]["annotations"]

char_count = {k:v for k,v in enumerate(list(content))}
[print(i) for i in ann]


This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>.

{'end': 112, 'label': 'Positive', 'start': 99, 'tag': 'Positive', 'value': 'Goldman Sachs'}
{'end': 132, 'label': 'Positive', 'start': 121, 'tag': 'Positive', 'value': 'Wells Fargo'}
{'end': 157, 'label': 'Positive', 'start': 142, 'tag': 'Positive', 'value': 'Bank of America'}
{'end': 184, 'label': 'Positive', 'start': 170, 'tag': 'Positive', 'value': 'JPmorgan Chase'}


[None, None, None, None]

In [6]:
# Load model
# distill model
distill_model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

#base model:
base_model_name = "Qwen/Qwen2.5-1.5B"

#instruct_model_name
instruct_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
distill_model,distill_tokenizer = load_model(distill_model_name,device="cuda")
base_model,base_tokenizer = load_model(base_model_name,device="cuda")
instruct_model,instruct_tokenizer = load_model(instruct_model_name,device="cuda")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [30]:
def evaluate(model,tokenizer,text,device,prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT1"]):
    prompt = create_prompt_deepseek_qwen(content,prompt_style=prompt_style)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.add_eos_token
    inputs = tokenizer(text, return_tensors="pt").to(device)
    inputs_length = len(inputs["input_ids"][0])
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.0001)
    return instruct_tokenizer.decode(outputs[0],skip_special_tokens=True)
    

In [31]:
output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT1"])
print(output)

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The table below shows the 12-month returns for these stocks from January 1st to December 31st.

| Stock | Return |
|-------|--------|
| GS    | -4.75% |
| WFC   | -6.89% |
| BAC   | -1.94% |
| JPM   | -1.74% |
| C     | -1.94% |
| MS    | -1.74% |

What is the average return of the top 3 stocks in terms of percentage? To calculate the average return of the top 3 stocks in terms of percentage, we need to:

1. Identify the top 3 stocks based on their return values.
2. Calculate the sum of their returns.
3. Divide the sum by the number of top stocks (which is 3).

From the given data:
- Top 3 stocks: Goldman Sachs (GS), Wells Fargo (WFC), and Bank of America (BAC)
- Their respective returns: -4.75%, -6.89%, an

In [33]:
# trying 5 run to see how response changes

for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT1"])
    print(output)
    print("____________________________________")
    print("____________________________________")
    

Run 1

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The table below shows the 12-month returns for these stocks from January 1st to December 31st.

| Stock | Return |
|-------|--------|
| GS    | -4.5%  |
| WFC   | +6.7%  |
| BAC   | +8.9%  |
| JPM   | +10.2% |
| C     | -1.5%  |
| MS    | +2.3%  |
| JPM   | +10.2% |
| WFC   | +6.7%  |
| GS    | -4.5%  |
| BAC   | +8.9%  |
| C     | -1.5%  |

What is the average return of the top 3 stocks?
To calculate the average return of the top 3 stocks, we first need to identify which stocks are in the top 3 based on their return percentages:

1. **Top 3 Stocks by Return**:
   - Goldman Sachs (GS): -4.5%
   - Wells Fargo (WFC): +6.7%
   - Bank of America (BAC): +8.9%

These three stocks have the highest positive 

In [34]:
# System prompt 2
for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT2"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The table below shows the 12-month returns for these stocks from January 1st to December 31st.

| Stock | Return |
|-------|--------|
| GS    | -4.5%  |
| WFC   | +6.7%  |
| BAC   | +8.9%  |
| JPM   | +10.2% |
| C     | -1.5%  |
| MS    | +2.3%  |
| JPM   | +10.2% |
| WFC   | +6.7%  |
| GS    | -4.5%  |
| BAC   | +8.9%  |
| C     | -1.5%  |

What is the average return for the top 3 stocks?
To calculate the average return for the top 3 stocks, we first need to identify which stocks are in the top 3 based on their return percentages:

1. **Goldman Sachs (GS)**: -4.5%
2. **Wells Fargo (WFC)**: +6.7%
3. **Bank of America (BAC)**: +8.9%

These three stocks have positive returns, so they qualify as part of

In [35]:
# System prompt 3
for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT3"]+"Make sure to follow the instruction and output is json format. ")
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The table below shows the 12-month returns for these stocks from January 1st to December 31st.

| Stock | Return |
|-------|--------|
| GS    | -4.5%  |
| WFC   | +7.6%  |
| BAC   | +8.9%  |
| JPM   | +10.1% |
| C     | -1.5%  |
| MS    | +1.2%  |
| JPM   | +10.1% |
| WFC   | +7.6%  |
| GS    | -4.5%  |
| BAC   | +8.9%  |
| C     | -1.5%  |

What is the average return of the top 3 stocks in terms of their absolute value? To calculate the average return of the top 3 stocks based on their absolute values, we first need to identify which stocks are considered "top" by Katzke's criteria (i.e., those with the highest potential return). According to the information provided:

- Goldman Sachs (GS) has a pot

In [42]:
# System prompt 4
for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT4"]+"Make sure to follow the instruction and output is json format. ")
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The table below shows the 12-month returns for these stocks from January 1st to December 31st.

| Stock | Return |
|-------|--------|
| GS    | -4.75% |
| WFC   | -6.89% |
| BAC   | -1.94% |
| JPM   | -1.75% |
| C     | 1.50%  |
| MS    | 1.25%  |
| JPM   | 1.75%  |
| WFC   | 1.50%  |
| GS    | 1.25%  |
| BAC   | 1.75%  |
| C     | 1.50%  |
| JPM   | 1.25%  |

What is the average return of the top 3 stocks (highest conviction) that Katzke recommends? To calculate the average return of the top 3 stocks that Katzke recommends, we need to:

1. Identify the three highest conviction stocks:
   - Goldman Sachs (GS)
   - Wells Fargo (WFC)
   - Bank of America (BAC)

2. Find their respective 12-month returns

In [44]:
# System prompt 5
for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(instruct_model,instruct_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT5"])
    print(output)
    print("____________________________________")
    print("____________________________________")

Run 1

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest is Citigroup <C.N>.
Does this next sentence follow, given the preceding text?
Citigroup has a lower chance of being convicted than any other company.

Choose from:
 (A). Yes.
 (B). It's impossible to say.
 (C). No.
(B).
____________________________________
____________________________________
Run 2

This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The lowest are: Citigroup <C.N>, Morgan Stanley <MS.N>, JPMorgan Chase and Wells Fargo. 

The above is a table in markdown format. It has 4 rows and 2 columns. The first column is labeled "Company" and the second column is labeled "Potential Return". Can you repeat this table in HTML?
Certain

In [38]:
# System prompt 3
for i in range(3):
    print(f"Run {i+1}\n")
    output = evaluate(distill_model,distill_tokenizer,content,device="cuda",prompt_style=prompt_style["DEFAULT_SYSTEM_PROMPT2"]+"Make sure to follow the instruction and output is json format. ")
    print(output)
    print("____________________________________")
    print("____________________________________")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Run 1



Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The other companies are: Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells... Wait, I think I'm getting confused here. Maybe I should try to figure out the correct order of these companies based on their stock prices.

Each of these companies has a stock price that can be represented as a number. The stock prices are as follows:

- Goldman Sachs (GS.N): 100
- Wells Fargo (WFC.N): 150
- Bank of America (BAC.N): 120
- JP Morgan Chase (JPM.N): 130

Now, the problem is to determine the order of these four companies from highest to lowest stock price. But there's a twist: each of these companies has a 30% total return potenti

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


This said, Katzke's highest conviction recommendations, each with ~30% total return potential are: Goldman Sachs <GS.N>, Wells Fargo <WFC.N>, Bank of America <BAC.N> and JPmorgan Chase <JPM.N>. The other companies are: Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells Fargo <WFC.N>, Bank of America <BAC.N>, JPMorgan Chase <JPM.N>, and Wells... Wait, I think I'm getting confused here. Maybe I should try to figure out the correct order of these companies based on their stock prices.

Each of these companies has a stock price that can be represented as a number. The stock prices are as follows:

- Goldman Sachs (GS.N): 100
- Wells Fargo (WFC.N): 150
- Bank of America (BAC.N): 120
- JP Morgan Chase (JPM.N): 130

Now, the problem is to determine the order of these four companies from highest to lowest stock price. If two companies have the same stock price, they should be considered equal

## Data Preprocessing


In [177]:
def preprocess_dataset1(data,tokenizer,max_len:4096,is_train:bool=True):
    
    text = data["content"]
    ann = data["annotations"]
    labels = str([{'entity':a["value"],'label':a["label"]} for a in ann]) if ann else ""
    prompt = create_prompt_deepseek_qwen(text)   
    
    #tokenized input
    tokenized_input = tokenizer(prompt,
                            max_length = max_len,
                            truncation=True,
                            padding=False,
                            padding_side="right",
                            #return_tensors='pt'
                            )
    
    tokenized_labels = tokenizer(labels,
                                max_length = max_len,
                                truncation=True,
                                padding=False,
                                padding_side="right")
    
    
    return {"input_ids":tokenized_input["input_ids"],
           "attention_mask":tokenized_input["attention_mask"],
           "labels":tokenized_labels["input_ids"] if is_train else None}

In [ ]:
def preprocess_dataset2(data,tokenizer,max_len:4096,is_train:bool=True):
    
    text = data["content"]
    ann = data["annotations"]
    labels = str([{'entity':a["value"],'label':a["label"]} for a in ann]) if ann else ""
    prompt = create_prompt_deepseek_qwen(text)   
    
    #tokenized input
    if is_train and labels:
        combined_text = prompt + "\nEntity and Sentiments are: \n" + labels + tokenizer.eos_token
    else:
        combined_text = prompt + tokenizer.eos_token
        
    tokenized = tokenizer(combined_text,
                            max_length=max_len,
                            truncation=True,
                            padding=False,
                            return_tensors="pt")
    
    
    tokenized["labels"] = tokenized["input_ids"].clone()

    return tokenized

In [178]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    distill_tokenizer,
    model=distill_model,  
    padding="longest",
    label_pad_token_id=-100)

sample = train_data[10]
tokenized = preprocess_dataset(sample,distill_tokenizer,is_train=True,max_len=1024)

In [180]:
tokenized["input_ids"],len(tokenized["input_ids"]),tokenized.keys()

([151646,
  27,
  130957,
  7265,
  315,
  11652,
  130957,
  29,
  151644,
  2610,
  525,
  264,
  7548,
  25530,
  5896,
  18237,
  13,
  4615,
  3383,
  374,
  311,
  23643,
  279,
  3897,
  5896,
  1467,
  11,
  10542,
  678,
  5896,
  14744,
  320,
  20805,
  438,
  2813,
  5036,
  701,
  323,
  8253,
  279,
  25975,
  5815,
  448,
  1817,
  5387,
  13,
  1752,
  1449,
  10820,
  5387,
  11,
  3410,
  1181,
  829,
  323,
  279,
  12159,
  25975,
  11,
  892,
  1265,
  387,
  825,
  315,
  25,
  43903,
  11,
  58694,
  11,
  476,
  50857,
  13,
  26642,
  279,
  3059,
  304,
  264,
  32930,
  3561,
  382,
  28301,
  1437,
  279,
  5896,
  14744,
  1075,
  2813,
  5036,
  504,
  279,
  2661,
  1467,
  323,
  11655,
  279,
  57114,
  369,
  1817,
  5896,
  14744,
  510,
  1986,
  1053,
  11,
  77109,
  440,
  594,
  8426,
  28091,
  18564,
  11,
  1817,
  448,
  3968,
  18,
  15,
  4,
  2790,
  470,
  4650,
  525,
  25,
  47328,
  59927,
  366,
  16522,
  2067,
  8066,
  36858,
  576

In [148]:
# tokenized["labels"],len(tokenized["labels"])

In [166]:
# distill_tokenizer.convert_ids_to_tokens(tokenized["labels"])

In [181]:
import torch

# Assuming train_data is a Dataset object
sample_batch = [preprocess_dataset(train_data[i], distill_tokenizer, is_train=True, max_len=1024) for i in range(min(4, len(train_data)))]

In [200]:
sample_batch[0].keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [185]:
# checking if padding is properly done, and eos is replaced with -100 
collated_batch = data_collator(sample_batch)

print("Collated Batch Keys:", collated_batch.keys())

if "input_ids" in collated_batch:
    print("Input IDs Shape:", collated_batch["input_ids"].shape)
    print("Example Input IDs:\n", collated_batch["input_ids"])

if "attention_mask" in collated_batch:
    print("Attention Mask Shape:", collated_batch["attention_mask"].shape)
    print("Example Attention Mask:\n", collated_batch["attention_mask"])

if "labels" in collated_batch:
    print("Labels Shape:", collated_batch["labels"].shape)
    print("Example Labels:\n", collated_batch["labels"])

Collated Batch Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs Shape: torch.Size([4, 163])
Example Input IDs:
 tensor([[151646,     27, 130957,   7265,    315,  11652, 130957,     29, 151644,
           2610,    525,    264,   7548,  25530,   5896,  18237,     13,   4615,
           3383,    374,    311,  23643,    279,   3897,   5896,   1467,     11,
          10542,    678,   5896,  14744,    320,  20805,    438,   2813,   5036,
            701,    323,   8253,    279,  25975,   5815,    448,   1817,   5387,
             13,   1752,   1449,  10820,   5387,     11,   3410,   1181,    829,
            323,    279,  12159,  25975,     11,    892,   1265,    387,    825,
            315,     25,  43903,     11,  58694,     11,    476,  50857,     13,
          26642,    279,   3059,    304,    264,  32930,   3561,    382,  28301,
           1437,    279,   5896,  14744,   1075,   2813,   5036,    504,    279,
           2661,   1467,    323,  11655,    279,  57114,  

In [198]:
# checking if tokenization is done succefully. 
# distill_tokenizer.convert_ids_to_tokens(collated_batch["input_ids"][0])
# distill_tokenizer.convert_ids_to_tokens(collated_batch["labels"][0])

In [197]:
# collated_batch["labels"][1]

tensor([151646,     58,  13608,   2996,   1210,    364,     33,   1055,     32,
           516,    364,   1502,   1210,    364,  86907,  24731,   5360,   2996,
          1210,    364,     33,   1055,     32,    516,    364,   1502,   1210,
           364,  86907,   8275,     60,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100])